# Cadena de Markov

Simula `M` veces una cadena de Markov de `N` estados con matriz de transición `P`
(cada fila suma 1) y pinta cada simulación como una tira de colores, un color por estado.
Edita la matriz y pulsa **Simular Cadenas**.

Librerías: `conda install -n profesia numpy matplotlib ipywidgets`

In [1]:
import ast

import ipywidgets as widgets
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display


# --- FUNCIÓN DE SIMULACIÓN MULTIPLE ---
def simulate_markov_multiple(P, initial_state, steps, M):
    """Simula una cadena de Markov M veces."""
    N = P.shape[0]
    results = np.zeros((M, steps + 1), dtype=int)
    results[:, 0] = initial_state

    for m in range(M):
        current = initial_state
        for t in range(1, steps + 1):
            current = np.random.choice(N, p=P[current])
            results[m, t] = current

    return results


# --- DEFINICIÓN DE LA INTERFAZ (WIDGETS) ---

N_input = widgets.IntSlider(value=3, min=2, max=10, description="N (Estados):")
M_input = widgets.IntSlider(value=5, min=1, max=15, description="M (Sims):")
steps_input = widgets.IntSlider(
    value=60, min=10, max=200, step=10, description="Pasos:"
)
initial_state_input = widgets.IntText(
    value=0, description="Est. Inic.:", layout=widgets.Layout(width="150px")
)

matrix_input = widgets.Textarea(
    description="Matriz P:", layout=widgets.Layout(width="400px", height="180px")
)

simulate_btn = widgets.Button(
    description="Simular Cadenas", button_style="success", icon="dna"
)

output_plot = widgets.Output()


def generate_uniform_matrix(n):
    """Texto de una matriz n × n equiprobable, para el cuadro «Matriz P»."""
    val = 1.0 / n
    mat = [[round(val, 4) for _ in range(n)] for _ in range(n)]
    return "[\n" + ",\n".join(["  " + str(row) for row in mat]) + "\n]"


# Inicializar la matriz y observar los cambios en el slider N
matrix_input.value = generate_uniform_matrix(N_input.value)


def on_N_change(change):
    """Regenera la matriz al cambiar N."""
    matrix_input.value = generate_uniform_matrix(change.new)
    # Asegurar que el estado inicial no supere el nuevo N
    if initial_state_input.value >= change.new:
        initial_state_input.value = 0


N_input.observe(on_N_change, names="value")

# --- LÓGICA DEL DASHBOARD ---


def update_dashboard(b):
    """Lee los controles, valida la matriz, simula y dibuja."""
    with output_plot:
        clear_output(wait=True)
        try:
            # 1. Leer y validar
            P = np.array(ast.literal_eval(matrix_input.value))
            N = P.shape[0]

            if P.ndim != 2 or P.shape[0] != P.shape[1]:
                raise ValueError("La matriz debe ser cuadrada (N x N).")
            if not np.allclose(P.sum(axis=1), 1, atol=1e-3):
                raise ValueError("Las filas de la matriz deben sumar 1.")
            if initial_state_input.value >= N or initial_state_input.value < 0:
                raise ValueError(f"El estado inicial debe estar entre 0 y {N - 1}.")

            steps = steps_input.value
            M = M_input.value
            init_state = initial_state_input.value

            # 2. Simular
            results = simulate_markov_multiple(P, init_state, steps, M)

            # 3. Graficar "Cromosomas"
            fig, axes = plt.subplots(M, 1, figsize=(12, 0.4 * M + 1))
            axes = np.atleast_1d(axes)  # Asegurar que es un array aunque M=1

            # Paleta de colores categórica (hasta 10 estados distintos)
            cmap = plt.get_cmap("tab10", N)

            # Coordenadas para dibujar los cuadraditos (malla)
            X, Y = np.meshgrid(np.arange(steps + 2), [0, 1])

            for i in range(M):
                Z = results[i : i + 1, :]
                # pcolormesh dibuja los cuadrados con bordes blancos (genes)
                axes[i].pcolormesh(
                    X,
                    Y,
                    Z,
                    cmap=cmap,
                    vmin=-0.5,
                    vmax=N - 0.5,
                    edgecolors="white",
                    linewidth=1.5,
                )

                # Ocultar ejes Y y marcos para dar aspecto de cromosoma flotante
                axes[i].set_yticks([])
                axes[i].set_ylabel(
                    f"Sim {i + 1}",
                    rotation=0,
                    labelpad=25,
                    va="center",
                    fontweight="bold",
                )
                for spine in axes[i].spines.values():
                    spine.set_visible(False)

                # Mostrar el eje X solo en el último cromosoma
                if i < M - 1:
                    axes[i].set_xticks([])
                else:
                    axes[i].set_xlabel("Tiempo (Pasos / Genes)", fontweight="bold")
                    axes[i].tick_params(
                        axis="x", length=0
                    )  # Ocultar las rayitas del tick

            # 4. Leyenda Global
            patches = [
                mpatches.Patch(color=cmap(i), label=f"Estado {i}") for i in range(N)
            ]
            fig.legend(
                handles=patches,
                loc="center right",
                bbox_to_anchor=(1.10, 0.5),
                title="Estados",
            )

            plt.subplots_adjust(right=0.9, hspace=0.3)
            plt.show()

        except Exception as e:
            print(f"Error en los parámetros: {e}")


simulate_btn.on_click(update_dashboard)

# --- MOSTRAR EL DASHBOARD ---

# Organizar visualmente los widgets
controles_izq = widgets.VBox(
    [N_input, M_input, steps_input, initial_state_input, simulate_btn]
)
ui = widgets.VBox(
    [
        widgets.HTML("<h2>🧬 Simulador de Cadenas de Markov (Vista Cromosómica)</h2>"),
        widgets.HTML(
            "<p>Al cambiar <b>N</b>, la matriz P se regenerará automáticamente. Edítala según necesites.</p>"
        ),
        widgets.HBox(
            [controles_izq, matrix_input],
            layout=widgets.Layout(align_items="center", grid_gap="20px"),
        ),
        output_plot,
    ]
)

display(ui)
update_dashboard(None)  # Render inicial